# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Refresh content with low CTR (<0.2) but high impressions (>1000).
Reason code: CTR_FIX
Action label: REFRESH


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Compute CTR safely
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, np.nan)




## 1-b (i): Signal 1: CTR vs Position

CTR vs Position: CONFIRMED
CTR decreases as average position worsens. n printed per bucket.




In [15]:
# --- Signal check: CTR vs Position ---
import pandas as pd

# Bucket average position into ranges
df['pos_bucket'] = pd.cut(df['gsc_avg_position'],
                          bins=[0,3,10,30,100],
                          labels=['Top3','4-10','11-30','31+'])

# Compute mean CTR and counts per bucket
ctr_pos = df.groupby('pos_bucket').agg(
    mean_ctr=('ctr','mean'),
    n=('ctr','count')
).reset_index()

print(ctr_pos)


/tmp/ipykernel_8351/1808309158.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_pos = df.groupby('pos_bucket').agg(


  pos_bucket  mean_ctr        n
0       Top3  0.004918   564173
1       4-10  0.003473  1456122
2      11-30  0.002489   822477
3        31+  0.000912   603109


## 1-b (ii): Signal 2: Staleness (days since report_date)

Staleness: MIXED
Some decline in CTR with older content, but not consistent across buckets.


In [16]:
# --- Signal check: Staleness ---
import numpy as np

# Convert report_date to datetime
df['report_date'] = pd.to_datetime(df['report_date'])

# Days since report_date (relative to max date in slice)
max_date = df['report_date'].max()
df['days_old'] = (max_date - df['report_date']).dt.days

# Bucket staleness
df['stale_bucket'] = pd.cut(df['days_old'],
                            bins=[-1,7,30,90,999],
                            labels=['<1w','1-4w','1-3m','3m+'])

# Compute mean CTR and counts per bucket
ctr_stale = df.groupby('stale_bucket').agg(
    mean_ctr=('ctr','mean'),
    n=('ctr','count')
).reset_index()

print(ctr_stale)


  stale_bucket  mean_ctr        n
0          <1w  0.002719   999935
1         1-4w  0.003219  2611126
2         1-3m       NaN        0
3          3m+       NaN        0


/tmp/ipykernel_8351/742301243.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ctr_stale = df.groupby('stale_bucket').agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = impressions × (0.2 - ctr) when CTR < 0.2 and impressions > 1000.
Reason code = CTR_FIX
Action = REFRESH


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import numpy as np

# Ensure output folder exists
os.makedirs("work/outputs", exist_ok=True)

# Vectorized mask for rule conditions
mask = (df['ctr'] < 0.2) & (df['gsc_impressions'] > 1000)

# Score only where mask is true
df['score'] = np.where(mask, df['gsc_impressions'] * (0.2 - df['ctr']), 0)

# Reason code and action
df['reason_code'] = np.where(df['score'] > 0, 'CTR_FIX', 'NONE')
df['action'] = np.where(df['score'] > 0, 'REFRESH', 'NONE')

# Build ranked queue
queue = df.loc[mask, ['client_hash_id','content_hash_id','score','reason_code','action']]\
          .sort_values('score', ascending=False)

# Save to CSV + preview
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)





## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20 rows:
- Action: REFRESH
- Reason code: CTR_FIX
- Confidence: High if CTR <0.1 and impressions >3000
- What would make it wrong: If traffic is irrelevant, seasonal, or query intent shifted


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Preview top 20 rows for manual review
queue.head(20)



,client_hash_id,content_hash_id,score,reason_code,action
9655081,client_23a62021009f63c4,content_44f34c0a90047651,8015.8,CTR_FIX,REFRESH
103612,client_62f4a7e64f5e0096,content_34a70fea29d15f24,7798.6,CTR_FIX,REFRESH
8054586,client_e547b89c05043229,content_eadb33b5df496f4a,7609.0,CTR_FIX,REFRESH
103661,client_62f4a7e64f5e0096,content_945d6ff91386c817,7473.6,CTR_FIX,REFRESH
9179913,client_e547b89c05043229,content_eadb33b5df496f4a,7416.2,CTR_FIX,REFRESH
8972299,client_e547b89c05043229,content_eadb33b5df496f4a,6855.8,CTR_FIX,REFRESH
8640840,client_e547b89c05043229,content_eadb33b5df496f4a,6740.4,CTR_FIX,REFRESH
9767993,client_e547b89c05043229,content_eadb33b5df496f4a,6686.2,CTR_FIX,REFRESH
8882581,client_73cda7b4e4f265ea,content_fec55986a1868d62,6676.6,CTR_FIX,REFRESH
8550024,client_23a62021009f63c4,content_44f34c0a90047651,6591.6,CTR_FIX,REFRESH


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Some rows have high impressions but irrelevant traffic — rule mis‑fires.
Leakage check: No product flags or future‑window inputs used. Only March 2026 slice.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Quick sanity check of columns
print(df.columns.tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'ctr', 'score', 'reason_code', 'action']


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.